# Statistical Analysis and Visualization of Daily Bike-Rental Demand
**Dataset:** UCI Bike Sharing Dataset (`data/raw/day.csv`, DOI: `10.24432/C5W894`)

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.load_validate import load_and_validate_data
from src.descriptive_eda import run_exploratory_data_analysis
from src.probability import run_probability_analysis
from src.hypothesis_tests import run_all_hypothesis_tests
from src.correlations import run_correlation_analysis
from src.regression import run_regression_analysis

## Phase 1: Dataset Provenance & Validation

In [2]:
df, audit = load_and_validate_data(save_outputs=False)
print('Dimensions:', audit['dimensions'], '| SHA-256 Verified:', audit['provenance']['day_csv_checksum_verified'])
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/data_dictionary_summary.csv'))

Dimensions: {'n_rows': 731, 'n_cols': 16} | SHA-256 Verified: True


,column,dtype,missing_count,unique_count,min,max,mean,std,median
0,instant,int64,0,731,1.0,731.0,366.0000,211.1658,366.0000
1,dteday,str,0,731,2011-01-01,2012-12-31,NaN,NaN,NaN
2,season,int64,0,4,1.0,4.0,2.4966,1.1108,3.0000
3,yr,int64,0,2,0.0,1.0,0.5007,0.5003,1.0000
4,mnth,int64,0,12,1.0,12.0,6.5198,3.4519,7.0000
5,holiday,int64,0,2,0.0,1.0,0.0287,0.1672,0.0000
6,weekday,int64,0,7,0.0,6.0,2.9973,2.0048,3.0000
7,workingday,int64,0,2,0.0,1.0,0.6840,0.4652,1.0000
8,weathersit,int64,0,3,1.0,3.0,1.3953,0.5449,1.0000
9,temp,float64,0,499,0.0591304,0.861667,0.4954,0.1831,0.4983


## Phase 2: Exploratory Data Analysis (EDA) & Outlier Audit

In [3]:
eda_res = run_exploratory_data_analysis(df, save_outputs=False)
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/eda_continuous_summary.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/eda_group_by_season.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/eda_outlier_audit.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/eda_outlier_handling_sensitivity.csv'))

,variable,unit_description,n,mean,sd,variance,se_mean,ci95_mean_low,ci95_mean_high,trimmed_mean_10pct,...,range,iqr,mad_median,cv_pct,skewness,excess_kurtosis,physical_mean,physical_sd,physical_median,physical_iqr
0,cnt,bikes/day,731,4504.3488,1937.2115,3.752788e+06,71.6504,4363.6835,4645.0142,4517.1897,...,8692.0000,2804.0000,1407.0000,43.0076,-0.0474,-0.8119,4504.35,1937.21,4548.00,2804.00
1,casual,bikes/day,731,848.1765,686.6225,4.714504e+05,25.3957,798.3192,898.0337,744.9504,...,3408.0000,780.5000,396.0000,80.9528,1.2665,1.3221,848.18,686.62,713.00,780.50
2,registered,bikes/day,731,3656.1724,1560.2564,2.434400e+06,57.7082,3542.8786,3769.4661,3641.7231,...,6926.0000,2279.5000,1155.0000,42.6746,0.0437,-0.7131,3656.17,1560.26,3662.00,2279.50
3,temp,"normalized [0,1] (×41°C)",731,0.4954,0.1831,3.350000e-02,0.0068,0.4821,0.5087,0.4972,...,0.8025,0.3183,0.1583,36.9513,-0.0545,-1.1189,20.31,7.51,20.43,13.05
4,atemp,"normalized [0,1] (×50°C)",731,0.4744,0.1630,2.660000e-02,0.0060,0.4625,0.4862,0.4769,...,0.7618,0.2708,0.1356,34.3543,-0.1311,-0.9851,23.72,8.15,24.34,13.54
5,hum,"normalized [0,1] (×100%)",731,0.6279,0.1424,2.030000e-02,0.0053,0.6176,0.6382,0.6271,...,0.9725,0.2102,0.1046,22.6836,-0.0698,-0.0645,62.79,14.24,62.67,21.02
6,windspeed,"normalized [0,1] (×67 km/h)",731,0.1905,0.0775,6.000000e-03,0.0029,0.1849,0.1961,0.1851,...,0.4851,0.0983,0.0491,40.6842,0.6773,0.4109,12.76,5.19,12.13,6.59


,group_variable,group_code,group_label,n,pct_of_days,mean,sd,se_mean,ci95_mean_low,ci95_mean_high,median,iqr,min,max
0,season,1,1: Spring (Dec 21–Mar 20),181,24.76,2604.1326,1399.9421,104.0569,2398.8044,2809.4608,2209.0,1918.00,431.0,7836.0
1,season,2,2: Summer (Mar 21–Jun 20),184,25.17,4992.3315,1695.9772,125.0291,4745.6476,5239.0154,4941.5,2374.00,795.0,8362.0
2,season,3,3: Fall (Jun 21–Sep 22),188,25.72,5644.3032,1459.8004,106.4669,5434.2727,5854.3337,5353.5,2342.75,1115.0,8714.0
3,season,4,4: Winter (Sep 23–Dec 20),178,24.35,4728.1629,1699.6153,127.3915,4476.7612,4979.5647,4634.5,2009.00,22.0,8555.0


,variable,q25,q75,iqr,tukey_lower_fence_1_5_iqr,tukey_upper_fence_1_5_iqr,iqr_outliers_below_count,iqr_outliers_above_count,iqr_outliers_total_count,iqr_outliers_pct,extreme_3_0_iqr_count,zscore_abs_gt_3_count,zscore_abs_gt_3_pct,max_abs_zscore,sample_iqr_outlier_dates,sample_z_outlier_dates
0,cnt,3152.0000,5956.0000,2804.0000,-1054.0000,10162.0000,0,0,0,0.00,0,0,0.00,2.3138,NaN,NaN
1,casual,315.5000,1096.0000,780.5000,-855.2500,2266.7500,0,44,44,6.02,0,8,1.09,3.7311,"2011-05-29, 2011-07-03, 2011-07-04, 2011-07-16...","2011-07-04, 2012-03-17, 2012-04-07, 2012-05-19..."
2,registered,2497.0000,4776.5000,2279.5000,-922.2500,8195.7500,0,0,0,0.00,0,0,0.00,2.3305,NaN,NaN
3,temp,0.3371,0.6554,0.3183,-0.1404,1.1329,0,0,0,0.00,0,0,0.00,2.3832,NaN,NaN
4,atemp,0.3378,0.6086,0.2708,-0.0683,1.0147,0,0,0,0.00,0,0,0.00,2.4256,NaN,NaN
5,hum,0.5200,0.7302,0.2102,0.2047,1.0455,2,0,2,0.27,0,2,0.27,4.4085,"2011-02-19, 2011-03-10","2011-02-19, 2011-03-10"
6,windspeed,0.1349,0.2332,0.0983,-0.0124,0.3806,0,13,13,1.78,0,2,0.27,4.0901,"2011-02-14, 2011-02-19, 2011-04-04, 2011-04-05...","2011-02-19, 2012-03-08"


,strategy,n_days,mean_cnt,sd_cnt,pearson_r_cnt_hum,pearson_p_cnt_hum,pearson_r_cnt_windspeed,pearson_p_cnt_windspeed
0,Untouched Full Dataset,731,4504.35,1937.21,-0.1007,0.006454,-0.2345,1.359959e-10
1,Drop Sensor Anomaly (hum > 0),730,4509.67,1933.19,-0.1146,0.001923,-0.2328,1.933490e-10
2,Winsorized Weather (1.5*IQR),731,4504.35,1937.21,-0.1057,0.004237,-0.2337,1.602171e-10


## Phase 3: Probability Distribution Analysis

In [4]:
prob_res = run_probability_analysis(df, save_outputs=False)
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/probability_distribution_summary.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/cnt_overdispersion_and_fit.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/temp_distribution_diagnostics.csv'))

,variable,mean,sd,variance,var_to_mean_ratio,skewness,excess_kurtosis,shapiro_w,shapiro_p,dagostino_k2,dagostino_p,lilliefors_ks,lilliefors_boot_p
0,cnt,4504.3488,1937.2115,3.752788e+06,833.1478,-0.0474,-0.8119,0.9801,2.081293e-08,62.7077,2.416595e-14,0.0471,0.0010
1,temp,0.4954,0.1831,3.350000e-02,0.0676,-0.0545,-1.1189,0.9659,5.145874e-12,294.2970,1.242282e-64,0.0727,0.0005
2,hum,0.6279,0.1424,2.030000e-02,0.0323,-0.0698,-0.0645,0.9933,2.480849e-03,0.6835,7.105427e-01,0.0339,0.0425
3,windspeed,0.1905,0.0775,6.000000e-03,0.0315,0.6773,0.4109,0.9712,8.426139e-11,51.4248,6.811539e-12,0.0627,0.0005


,lambda_hat,sample_variance,variance_to_mean_ratio_index_of_dispersion,dispersion_chi2_stat,dispersion_chi2_pvalue,cameron_trivedi_t_stat,cameron_trivedi_pvalue,log_likelihood,aic,ks_distance,negbin_r,negbin_p,negbin_loglik,negbin_aic,normal_loglik,normal_aic
0,4504.3488,3.752788e+06,833.1478,608197.88,0.0,24.7853,3.541296e-99,-338099.72,676201.44,0.4576,5.4129,0.0012,-6632.31,13268.61,-6569.69,13143.37


,variable,mean,sd,skewness,excess_kurtosis,mu_hat,sigma_hat,log_likelihood,aic,shapiro_wilk_stat,...,beta_aic,beta_ks_distance,season_1_mean,season_1_sd,season_2_mean,season_2_sd,season_3_mean,season_3_sd,season_4_mean,season_4_sd
0,temp,0.4954,0.1831,-0.0545,-1.1189,0.4954,0.1831,204.49,-404.97,0.9659,...,-464.7,0.0614,0.2977,0.1027,0.5444,0.1226,0.7063,0.0708,0.4229,0.1078


## Phase 4: Hypothesis Testing (H1 t-Test, H2 ANOVA, H3 Chi-Square)

In [5]:
ht_res = run_all_hypothesis_tests(df, save_outputs=False)
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/hypothesis_tests_summary.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/h1_ttest_workingday.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/h2_anova_season_posthoc.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/h3_chisq_contingency_table.csv'))

,hypothesis_id,test_family,variables,test_statistic,p_value_unadjusted,p_value_holm_adjusted,effect_size,decision_alpha_0_05
0,H1,Welch Two-Sample t-Test,workingday (0 vs 1) -> cnt,t_Welch = 1.6014 (df = 413.94),1.100574e-01,1.100574e-01,Hedges' g = +0.1315,Fail to Reject H0
1,H2,One-Way & Welch ANOVA,season (1..4) -> cnt,"F(3, 727) = 128.7696",6.720391e-67,2.016117e-66,"eta^2 = 0.3470, omega^2 = 0.3440",Reject H0
2,H3,Pearson Chi-Square Independence,weathersit (1..3) x high_demand (cnt > 4548),chi^2(2) = 44.0879,2.669494e-10,5.338989e-10,Cramer's V = 0.2456,Reject H0


,segment,n1,n0,mean1,sd1,median1,iqr1,mean0,sd0,median0,...,brown_forsythe_median_f,brown_forsythe_median_p,shapiro_group1_w,shapiro_group1_p,shapiro_group0_w,shapiro_group0_p,mann_whitney_u,p_value_mann_whitney,rank_biserial_r,p_value_permutation_10000
0,cnt,500.0,231.0,4584.82,1878.4156,4582.0,2643.25,4330.1688,2052.1412,4459.0,...,4.3454,0.037457,0.9766,3.489374e-07,0.9761,0.000603,61893.0,0.118605,0.0717,0.0987
1,registered,NaN,NaN,3978.25,NaN,NaN,NaN,2959.0346,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,casual,NaN,NaN,606.57,NaN,NaN,NaN,1371.1342,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,group1,group1_label,group2,group2_label,mean_group1,mean_group2,mean_diff_g2_minus_g1,tukey_ci95_low,tukey_ci95_high,tukey_p_adj,tukey_reject_0_05,welch_t_g2_minus_g1,welch_df,welch_p_unadjusted,hedges_g_g2_minus_g1,welch_p_holm_adjusted,welch_reject_0_05
0,1,1: Spring (UCI 1),2,2: Summer (UCI 2),2604.1326,4992.3315,2388.1989,1965.3325,2811.0653,0.0000,True,14.6816,352.4142,2.062553e-38,1.5314,1.031277e-37,True
1,1,1: Spring (UCI 1),3,3: Fall (UCI 3),2604.1326,5644.3032,3040.1706,2619.5409,3460.8003,0.0000,True,20.4213,366.9946,1.827511e-62,2.1205,1.096507e-61,True
2,1,1: Spring (UCI 1),4,4: Winter (UCI 4),2604.1326,4728.1629,2124.0303,1697.6444,2550.4163,0.0000,True,12.9129,342.1923,2.390012e-31,1.3624,9.560050e-31,True
3,2,2: Summer (UCI 2),3,3: Fall (UCI 3),4992.3315,5644.3032,651.9717,233.0927,1070.8507,0.0004,True,3.9702,359.5879,8.675074e-05,0.4115,1.735015e-04,True
4,2,2: Summer (UCI 2),4,4: Winter (UCI 4),4992.3315,4728.1629,-264.1686,-688.8276,160.4904,0.3782,False,-1.4800,359.5497,1.397575e-01,-0.1553,1.397575e-01,False
5,3,3: Fall (UCI 3),4,4: Winter (UCI 4),5644.3032,4728.1629,-916.1403,-1338.5720,-493.7085,0.0000,True,-5.5181,349.3088,6.686180e-08,-0.5783,2.005854e-07,True


,weathersit,weathersit_label,observed_low_demand,observed_high_demand,row_total,expected_low_demand,expected_high_demand,pct_low_demand,pct_high_demand,pearson_residual_high_demand,std_adjusted_residual_high_demand
0,1,1: Clear / Partly Cloudy,193,270,463,231.8167,231.1833,41.68,58.32,2.5529,5.9587
1,2,2: Mist / Cloudy,153,94,247,123.6689,123.3311,61.94,38.06,-2.6411,-4.5872
2,3,3: Light Snow / Light Rain,20,1,21,10.5144,10.4856,95.24,4.76,-2.9293,-4.2007


## Phase 5: Pearson & Spearman Correlation Analysis

In [6]:
corr_res = run_correlation_analysis(df, save_outputs=False)
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/pearson_correlation_matrix.csv', index_col=0))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/spearman_correlation_matrix.csv', index_col=0))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/correlation_pairwise_tests.csv'))

,cnt,temp,atemp,hum,windspeed
cnt,1.0000,0.6275,0.6311,-0.1007,-0.2345
temp,0.6275,1.0000,0.9917,0.1270,-0.1579
atemp,0.6311,0.9917,1.0000,0.1400,-0.1836
hum,-0.1007,0.1270,0.1400,1.0000,-0.2485
windspeed,-0.2345,-0.1579,-0.1836,-0.2485,1.0000


,cnt,temp,atemp,hum,windspeed
cnt,1.0000,0.6220,0.6228,-0.0981,-0.2172
temp,0.6220,1.0000,0.9926,0.1299,-0.1472
atemp,0.6228,0.9926,1.0000,0.1396,-0.1690
hum,-0.0981,0.1299,0.1396,1.0000,-0.2390
windspeed,-0.2172,-0.1472,-0.1690,-0.2390,1.0000


,var1,var2,pair_label,n,pearson_r,pearson_r_squared,pearson_ci95_low,pearson_ci95_high,pearson_p_unadjusted,spearman_rho,spearman_ci95_low,spearman_ci95_high,spearman_p_unadjusted,diff_rho_minus_r,magnitude_interpretation,pearson_p_holm,spearman_p_holm,significant_at_0_05
0,cnt,temp,cnt vs temp,731,0.6275,0.3937,0.5814,0.6695,2.810622e-81,0.6220,0.5755,0.6646,1.674555e-79,-0.0055,Strong (0.50 - 0.69),2.248498e-80,1.339644e-78,True
1,cnt,atemp,cnt vs atemp,731,0.6311,0.3982,0.5853,0.6728,1.854504e-82,0.6228,0.5763,0.6653,9.615484e-80,-0.0083,Strong (0.50 - 0.69),1.669054e-81,8.653935e-79,True
2,cnt,hum,cnt vs hum,731,-0.1007,0.0101,-0.1719,-0.0284,6.454143e-03,-0.0981,-0.1694,-0.0258,7.933230e-03,0.0025,Weak (0.10 - 0.29),6.454143e-03,7.933230e-03,True
3,cnt,windspeed,cnt vs windspeed,731,-0.2345,0.0550,-0.3019,-0.1648,1.359959e-10,-0.2172,-0.2852,-0.1470,2.969427e-09,0.0173,Weak (0.10 - 0.29),8.159752e-10,1.781656e-08,True
4,temp,atemp,temp vs atemp,731,0.9917,0.9835,0.9904,0.9928,0.000000e+00,0.9926,0.9914,0.9936,0.000000e+00,0.0009,Very Strong / Collinear (>= 0.70),0.000000e+00,0.000000e+00,True
5,temp,hum,temp vs hum,731,0.1270,0.0161,0.0550,0.1977,5.800888e-04,0.1299,0.0579,0.2005,4.297396e-04,0.0029,Weak (0.10 - 0.29),1.160178e-03,8.594793e-04,True
6,temp,windspeed,temp vs windspeed,731,-0.1579,0.0249,-0.2278,-0.0864,1.786857e-05,-0.1472,-0.2173,-0.0754,6.509802e-05,0.0108,Weak (0.10 - 0.29),7.147430e-05,2.603921e-04,True
7,atemp,hum,atemp vs hum,731,0.1400,0.0196,0.0682,0.2104,1.464024e-04,0.1396,0.0678,0.2100,1.520085e-04,-0.0003,Weak (0.10 - 0.29),4.392071e-04,4.560256e-04,True
8,atemp,windspeed,atemp vs windspeed,731,-0.1836,0.0337,-0.2528,-0.1126,5.755774e-07,-0.1690,-0.2386,-0.0977,4.345037e-06,0.0147,Weak (0.10 - 0.29),2.877887e-06,2.172518e-05,True
9,hum,windspeed,hum vs windspeed,731,-0.2485,0.0617,-0.3153,-0.1792,9.487956e-12,-0.2390,-0.3062,-0.1694,5.907300e-11,0.0095,Weak (0.10 - 0.29),6.641569e-11,4.135110e-10,True


## Phase 6: Multiple Linear Regression (OLS) & Diagnostics

In [7]:
reg_res = run_regression_analysis(df, save_outputs=False)
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/regression_model_comparison.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/regression_vif_diagnostics.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/regression_ols_coefficients.csv'))
display(pd.read_csv(PROJECT_ROOT / 'outputs/tables/regression_residual_diagnostics.csv'))

,model_name,role,n_predictors,train_mae,train_rmse,train_r2,train_adj_r2,test_mae,test_rmse,test_r2,test_bias_pred_minus_true
0,Mean Baseline (DummyRegressor),"Predicts Train Mean = 4,153.60",0,1442.9507,1787.3553,0.0000,0.0000,2290.4127,2560.5558,-0.8657,-1744.2034
1,Primary Multiple Linear Regression (OLS),Primary Specification (Drops Collinear atemp),11,539.4466,717.8453,0.8387,0.8356,891.6901,1164.8739,0.6139,-204.6966
2,Supplemental Collinear OLS (With atemp),Demonstrates VIF Collinearity,12,538.7984,717.6476,0.8388,0.8354,894.3660,1165.6082,0.6134,-209.0102
3,Supplemental Ridge (alpha=0.01),L2 Regularized Comparison,11,539.4474,717.8453,0.8387,0.8356,891.7117,1164.8879,0.6139,-204.7504


,feature,readable_name,vif,tolerance_1_over_vif,vif_when_atemp_included
0,num__temp,Normalized Temperature (temp),3.6091,0.2771,184.8565
1,num__hum,Normalized Humidity (hum),1.9176,0.5215,1.9374
2,num__windspeed,Normalized Wind Speed (windspeed),1.1846,0.8442,1.2463
3,bin__yr,Year Indicator (yr: 1 = 2012),1.2098,0.8266,1.2108
4,bin__holiday,Public Holiday Indicator (holiday: 1 = Yes),1.0702,0.9344,1.0786
5,bin__workingday,Working Day Indicator (workingday: 1 = Workday),1.0758,0.9295,1.0765
6,cat__season_2,Season 2: Summer (vs Season 1),2.6676,0.3749,2.6681
7,cat__season_3,Season 3: Fall (vs Season 1),4.6288,0.2160,4.7060
8,cat__season_4,Season 4: Winter (vs Season 1),1.7834,0.5607,1.7902
9,cat__weathersit_2,Weather 2: Mist/Cloudy (vs Clear 1),1.5861,0.6305,1.5937


,term,readable_name,coef_estimate,standardized_beta,se_iid,t_iid,p_iid,ci95_iid_low,ci95_iid_high,se_hc3,t_hc3,p_hc3,ci95_hc3_low,ci95_hc3_high,se_hac_lag7,t_hac_lag7,p_hac_lag7,ci95_hac_low,ci95_hac_high
0,const,"Intercept (Ref: 2011, Season 1, Clear Weather,...",1722.2094,NaN,217.7953,7.9075,1.361124e-14,1294.4334,2149.9854,216.5867,7.9516,1.841273e-15,1297.7073,2146.7115,256.6052,6.7115,1.926157e-11,1219.2724,2225.1463
1,num__temp,Normalized Temperature (temp),4714.3061,0.4962,303.0779,15.5548,9.309262e-46,4119.0248,5309.5875,348.7680,13.5170,1.240923e-41,4030.7334,5397.8789,583.1338,8.0844,6.245385e-16,3571.3850,5857.2273
2,num__hum,Normalized Humidity (hum),-1218.4637,-0.1008,281.2316,-4.3326,1.740655e-05,-1770.8363,-666.0912,289.4301,-4.2099,2.555154e-05,-1785.7363,-651.1911,322.7770,-3.7749,1.600466e-04,-1851.0951,-585.8324
3,num__windspeed,Normalized Wind Speed (windspeed),-2675.8292,-0.1140,429.1788,-6.2348,8.793267e-10,-3518.7879,-1832.8705,447.3011,-5.9822,2.201912e-09,-3552.5233,-1799.1351,512.7843,-5.2182,1.806361e-07,-3680.8680,-1670.7904
4,bin__yr,Year Indicator (yr: 1 = 2012),1947.8489,0.5276,68.1923,28.5640,3.453603e-112,1813.9110,2081.7868,72.5821,26.8365,1.212230e-158,1805.5906,2090.1071,105.4325,18.4748,3.291436e-76,1741.2050,2154.4927
5,bin__holiday,Public Holiday Indicator (holiday: 1 = Yes),-451.1620,-0.0412,190.2186,-2.3718,1.803131e-02,-824.7741,-77.5499,238.0091,-1.8956,5.801749e-02,-917.6513,15.3274,217.4627,-2.0747,3.801775e-02,-877.3811,-24.9428
6,bin__workingday,Working Day Indicator (workingday: 1 = Workday),66.8181,0.0174,67.0156,0.9971,3.191608e-01,-64.8086,198.4448,75.4826,0.8852,3.760424e-01,-81.1251,214.7612,78.3983,0.8523,3.940534e-01,-86.8398,220.4760
7,cat__season_2,Season 2: Summer (vs Season 1),1162.9026,0.3022,105.5280,11.0198,9.663524e-26,955.6329,1370.1722,120.7774,9.6285,6.062241e-22,926.1832,1399.6220,207.0331,5.6170,1.943135e-08,757.1252,1568.6800
8,cat__season_3,Season 3: Fall (vs Season 1),751.7570,0.1800,150.8922,4.9821,8.345057e-07,455.3866,1048.1274,170.4480,4.4105,1.031425e-05,417.6852,1085.8289,285.6124,2.6321,8.486174e-03,191.9671,1311.5470
9,cat__season_4,Season 4: Winter (vs Season 1),1465.4228,0.2947,111.5248,13.1399,1.185245e-34,1246.3748,1684.4709,96.4923,15.1869,4.316089e-52,1276.3014,1654.5443,146.5403,10.0001,1.521898e-23,1178.2091,1752.6366


,train_n,test_n,ols_f_statistic_iid,ols_f_pvalue_iid,ols_aic,ols_bic,durbin_watson,ljung_box_lag1_q,ljung_box_lag1_p,ljung_box_lag7_q,...,shapiro_wilk_w,shapiro_wilk_p,residual_skewness,residual_excess_kurtosis,residual_kurtosis,cooks_d_threshold_4_over_n,cooks_d_exceeding_4_over_n_count,cooks_d_max,cooks_d_max_date,worst_test_error_day
0,584,147,270.3766,2.488956e-218,9362.38,9414.82,1.2253,87.6334,7.878170e-21,145.7213,...,0.9764,4.319637e-08,-0.4031,1.6691,4.6446,0.006849,37,0.058992,2012-04-22,"{'dteday': '2012-10-30', 'actual_cnt': 1096, '..."
